# Day 01 · Points — The Doors

**Question.** Which Chicago 'L' stations can a rider who cannot take the stairs actually use?

**Approach.** Take every 'L' stop from the CTA's open data, collapse stops to stations, classify each
station as step-free (elevator or ramp) or stairs only, and project the stations into Chicago
address-grid coordinates for the plate.

**Output.** `output/plate.json`, read by the website to draw the plate.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, run settings |
| 2. Pull | Dated snapshot of the source data (cached) |
| 3. Validate | Schema, values and consistency checks that stop the run on surprises |
| 4. Transform | Stops → stations, line membership, address coordinates |
| 5. Analyze | Counts behind the finding, cross-checked against the CTA's published figure |
| 6. QA view | A quick chart to catch errors before publishing |
| 7. Export | `plate.json` with the data ledger |

Run top to bottom. To refresh the source data, set `REFRESH = True` in section 1.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from atlas import Ledger, snapshot, to_address, write_plate

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day01')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = pull a new snapshot from the Chicago Data Portal

DAY = 1
DATASET_ID = '8pix-ypme'  # CTA - System Information - List of 'L' Stops
DATASET_NAME = "CTA - System Information - List of 'L' Stops"
DATASET_URL = f'https://data.cityofchicago.org/d/{DATASET_ID}'

# Paths: the notebook runs from its own folder (days/day01-points/)
DAY_DIR = Path.cwd()
RAW_DIR = DAY_DIR / 'data' / 'raw'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
assert (DAY_DIR / 'day01.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

# Domain constants -----------------------------------------------------------------------------
LINES = ('Red', 'Blue', 'Brown', 'Green', 'Orange', 'Purple', 'Pink', 'Yellow')
CHICAGO_BBOX = {'lat': (41.60, 42.10), 'lon': (-87.95, -87.52)}  # generous: includes Evanston, O'Hare

# Published reference for the cross-check in section 5 (CTA, at the Austin station groundbreaking)
CTA_PUBLISHED = {'accessible': 104, 'total': 148}

## 2. Pull

The SODA API returns **one row per stop**, which usually means one row per platform direction, so a
station appears more than once. `snapshot` saves a dated CSV in `data/raw/` and reuses the latest
one on later runs, which keeps the published numbers reproducible.

In [ ]:
raw, pulled = snapshot(
    DATASET_ID,
    RAW_DIR,
    'l_stops',
    refresh=REFRESH,
    select='map_id,station_descriptive_name,ada,location',
)
log.info('Snapshot pulled %s: %d rows x %d columns', pulled, *raw.shape)
raw.head()

## 3. Validate

Each check guards an assumption the analysis relies on. If the portal changes its schema or
values, the run stops here instead of producing a wrong plate.

In [ ]:
EXPECTED = {'map_id', 'station_descriptive_name', 'ada', 'location.latitude', 'location.longitude'}
missing = EXPECTED - set(raw.columns)
assert not missing, f'Missing columns: {missing}'

stops = raw[list(EXPECTED)].copy()
stops['ada'] = stops['ada'].str.lower().str.strip()
stops['lat'] = pd.to_numeric(stops['location.latitude'], errors='coerce')
stops['lon'] = pd.to_numeric(stops['location.longitude'], errors='coerce')

checks = {
    'no missing station id': stops['map_id'].notna().all(),
    'no missing name': stops['station_descriptive_name'].notna().all(),
    'ADA flag is true/false': stops['ada'].isin({'true', 'false'}).all(),
    'coordinates parse': stops[['lat', 'lon']].notna().all().all(),
    'latitude inside Chicago area': stops['lat'].between(*CHICAGO_BBOX['lat']).all(),
    'longitude inside Chicago area': stops['lon'].between(*CHICAGO_BBOX['lon']).all(),
}
for name, ok in checks.items():
    log.info('%-32s %s', name, 'ok' if ok else 'FAILED')
assert all(checks.values()), 'Validation failed; see log above'

In [ ]:
# A station's stops should agree on name. The ADA flag can legitimately differ between platforms,
# so mixed stations are reported here and resolved conservatively in section 4.
per_station = stops.groupby('map_id').agg(
    names=('station_descriptive_name', 'nunique'),
    ada_values=('ada', 'nunique'),
    n_stops=('ada', 'size'),
)
assert (per_station['names'] == 1).all(), 'A station has more than one name'
mixed = per_station.index[per_station['ada_values'] > 1].tolist()
log.info('Stations: %d | stops per station: %s | mixed ADA flags: %s',
         len(per_station), per_station['n_stops'].value_counts().to_dict(), mixed or 'none')

## 4. Transform

- **Stops → stations.** A station counts as step-free only if *every* one of its stops is flagged
  accessible. A rider needs to reach the platform for their direction, so a single stair-only
  platform makes the station unreliable.
- **Lines.** Parsed from the parenthetical in the station name, e.g. `Belmont (Red, Brown & Purple lines)`.
  Interchange stations count once for each line they serve.
- **Coordinates.** Projected to Chicago address units (`atlas.grid`), the shared coordinate system
  of every plate.

In [ ]:
PAREN = re.compile(r'\((?P<inside>[^)]*)\)\s*$')


def parse_lines(station_name: str) -> list[str]:
    """Return the 'L' lines named in the station's parenthetical, in canonical order."""
    match = PAREN.search(station_name)
    inside = match.group('inside') if match else ''
    return [line for line in LINES if re.search(rf'\b{line}\b', inside)]


def short_name(station_name: str) -> str:
    """Station name without the line parenthetical: 'Clark/Lake (Blue, ...)' -> 'Clark/Lake'."""
    return PAREN.sub('', station_name).strip()


def collapse_to_stations(stops: pd.DataFrame) -> pd.DataFrame:
    """One row per station: step-free only if all stops are accessible; mean stop position."""
    stations = (
        stops.assign(step_free=stops['ada'].eq('true'))
        .groupby('map_id', as_index=False)
        .agg(
            name=('station_descriptive_name', 'first'),
            step_free=('step_free', 'all'),
            lat=('lat', 'mean'),
            lon=('lon', 'mean'),
        )
    )
    stations['name'] = stations['name'].str.replace(r'\s+', ' ', regex=True).str.strip()
    stations['short'] = stations['name'].map(short_name)
    stations['lines'] = stations['name'].map(parse_lines)
    stations['x'], stations['y'] = to_address(stations['lat'], stations['lon'])
    return stations.sort_values('map_id', ignore_index=True)


stations = collapse_to_stations(stops)
unparsed = stations.loc[stations['lines'].str.len() == 0, 'name'].tolist()
assert not unparsed, f'No line parsed for: {unparsed}'
stations.head()

## 5. Analyze

In [ ]:
total = len(stations)
stair_only = stations.loc[~stations['step_free']]

by_line = (
    stations.explode('lines')
    .groupby('lines')
    .agg(total=('map_id', 'size'), no_step_free=('step_free', lambda s: int((~s).sum())))
    .reindex(LINES)
    .rename_axis('line')
    .reset_index()
)
by_line['share_stair_only'] = (by_line['no_step_free'] / by_line['total']).round(3)

blue = by_line.set_index('line').loc['Blue']
summary = {
    'total': total,
    'noStepFree': len(stair_only),
    'blueTotal': int(blue['total']),
    'blueNoStepFree': int(blue['no_step_free']),
}
blue_share = summary['blueNoStepFree'] / summary['noStepFree']
log.info('Summary: %s | Blue share of stair-only stations: %.0f%%', summary, 100 * blue_share)
by_line.sort_values('no_step_free', ascending=False)

In [ ]:
# Cross-check against the CTA's own published count. A small gap is expected: the portal's ADA flag
# can lag stations reopened under the All Stations Accessibility Program, and the CTA counts
# stations slightly differently. A large gap means something is wrong with the data or the method.
portal_accessible = total - summary['noStepFree']
gap = {
    'accessible': portal_accessible - CTA_PUBLISHED['accessible'],
    'total': total - CTA_PUBLISHED['total'],
}
log.info('Portal %d of %d accessible vs CTA %d of %d (gap %s)',
         portal_accessible, total, CTA_PUBLISHED['accessible'], CTA_PUBLISHED['total'], gap)
assert all(abs(v) <= 6 for v in gap.values()), 'Portal and CTA counts diverge; investigate before publishing'

In [ ]:
# The finding printed on the plate, generated from the data so it can never drift from the numbers.
n_stair, n_total, n_blue = summary['noStepFree'], summary['total'], summary['blueNoStepFree']
half = 'Half' if abs(blue_share - 0.5) < 0.005 else f'{blue_share:.0%}'
FINDING = (
    f'{n_stair} of {n_total} \u2018L\u2019 stations still have no step-free entrance. '
    f'{half} of them, {n_blue}, are on one line: the Blue.'
)
print(FINDING)
stair_only[['map_id', 'short', 'lines']].reset_index(drop=True)

## 6. QA view

A plain check, not the final design: stations in address space (rings = stairs only) and the
stair-only count for each line. Look for stations far off the network or a line with an implausible count.

In [ ]:
fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(11, 7), gridspec_kw={'width_ratios': [1, 1.1]})

ax_map.scatter(stations.loc[stations['step_free'], 'x'], stations.loc[stations['step_free'], 'y'],
               s=12, color='0.15', label='step-free')
ax_map.scatter(stair_only['x'], stair_only['y'], s=22, facecolor='white', edgecolor='0.15',
               linewidth=1.1, label='stairs only')
ax_map.axhline(0, color='0.7', lw=0.6)
ax_map.axvline(0, color='0.7', lw=0.6)
ax_map.set_aspect('equal')
ax_map.set(title='Stations in address units', xlabel='west (-) / east (+)', ylabel='south (-) / north (+)')
ax_map.legend(loc='lower left', frameon=False)

order = by_line.sort_values('no_step_free')
ax_bar.barh(order['line'], order['total'], color='0.85', label='all stations')
ax_bar.barh(order['line'], order['no_step_free'], color='0.15', label='stairs only')
ax_bar.set(title='Stations by line', xlabel='stations')
ax_bar.legend(loc='lower right', frameon=False)
for side in ('top', 'right'):
    ax_bar.spines[side].set_visible(False)

fig.tight_layout()
plt.show()

## 7. Export

`plate.json` carries the data ledger, the finding and the station table. Keys are camelCase
because the website reads them directly.

In [ ]:
ledger = Ledger(
    source=DATASET_NAME,
    dataset_id=DATASET_ID,
    url=DATASET_URL,
    pulled=pulled,
    units=f'{total} stations (map_id); ADA flag as published',
    method=[
        'Stops collapsed to stations; step-free only if every stop is flagged accessible',
        'Plotted in Chicago address units: 800 per mile; 1,200 for the first mile south',
        'Origin: State & Madison',
    ],
    caveats=['ADA flag can lag stations reopened under the All Stations Accessibility Program'],
)

payload = {
    'source': {'name': DATASET_NAME, 'id': DATASET_ID, 'url': DATASET_URL},
    'summary': summary,
    'byLine': [
        {'line': r.line, 'total': int(r.total), 'noStepFree': int(r.no_step_free)}
        for r in by_line.itertuples()
    ],
    'stations': [
        {
            'id': r.map_id, 'name': r.name, 'short': r.short, 'lines': r.lines,
            'ada': bool(r.step_free), 'lat': round(r.lat, 6), 'lon': round(r.lon, 6),
            'x': round(float(r.x), 1), 'y': round(float(r.y), 1),
        }
        for r in stations.itertuples()
    ],
}

path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)

# Read back and confirm the contract the website depends on.
written = json.loads(path.read_text())
assert {'asOf', 'finding', 'ledger', 'summary', 'byLine', 'stations'} <= written.keys()
assert len(written['stations']) == total
log.info('Wrote %s (%.1f KB)', path.relative_to(DAY_DIR), path.stat().st_size / 1024)

## Notes for the plate

- Finding: printed in section 5; it is generated from the data.
- Caveat for the ledger: the portal's ADA flag can lag recent reopenings (for example Racine on the
  Blue Line), so the count of stair-only stations is an upper bound.
- Next: the website reads `output/plate.json`; rebuild the site and export the PNG.